# Hệ thống Kiểm tra Bất thường Bo mạch Cảm biến LM393 (Demo Độc lập trên Google Colab)

Notebook này chạy **trực tiếp toàn bộ giao diện kiểm tra ngay bên trong Google Colab** (không cần Streamlit, không cần mở Tunnel/Port ra ngoài, không cần cài đặt môi trường Python trên máy cá nhân), phục vụ đội **Kỹ thuật Điện** kiểm thử và đánh giá hệ thống phát hiện lỗi bo mạch **LM393**:

- **Tiền xử lý (`BoardAligner`)**: Tách nền chống bóng đổ (flat-field + multi-Otsu), xoay thẳng `minAreaRect`, khóa hướng 180°, mở rộng vùng lấy đủ 3 chân cắm và cân bằng sáng Gray-World + CLAHE.
- **Kiểm tra bề mặt bo mạch (`PatchCore` - `wide_resnet50_2`)**: Phát hiện thiếu/lệch/che khuất linh kiện SMD, biến trở `103`, IC `LM393`, đèn LED kèm **Bản đồ nhiệt (Anomaly Heatmap)**.
- **Kiểm tra hình học 3 chân cắm (`PinInspector`)**: Đo độc lập số lượng chân, chiều dài, độ rộng, góc nghiêng và khoảng cách của 3 chân `G (Trái)`, `V+ (Giữa)`, `S (Phải)`.
- **Thu nhận Camera (`CameraTrigger` & Webcam Colab)**: Hỗ trợ cả mô phỏng Auto-Trigger `10 frames` lẫn **chụp trực tiếp từ Webcam máy tính ngay trên trình duyệt Colab**.

---

### Hướng dẫn chạy trên Google Colab (Chỉ 2 bước)
1. *(Khuyên dùng để bật tốc độ nhanh nhất)*: Vào menu **Runtime > Change runtime type** → chọn **T4 GPU** (nếu để **CPU** hệ thống vẫn chạy bình thường).
2. Bấm **Runtime > Run all** (hoặc chạy lần lượt từ Cell 1):
   - Ngay tại **Cell 1**, nút **`Choose Files`** sẽ hiện ra để bạn chọn tải file **`colab_bundle.zip`** từ máy tính lên.
   - *(Mẹo tải nhanh hơn)*: Bạn cũng có thể bấm biểu tượng **`📁 Files`** ở thanh bên trái Colab rồi kéo thả `colab_bundle.zip` vào đó trước khi bấm **Run all** (hoặc đặt `colab_bundle.zip` trong Google Drive).

In [ ]:
# ==============================================================================
# 1. TẢI GÓI DỰ ÁN (colab_bundle.zip) TRÊN GOOGLE COLAB & NẠP MÔ HÌNH
# ==============================================================================
from __future__ import annotations

import base64
import os
import shutil
import sys
import zipfile
from datetime import datetime
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import HTML, clear_output, display

try:
    from google.colab import files as colab_files  # type: ignore[import-not-found]
    from google.colab.output import eval_js as colab_eval_js  # type: ignore[import-not-found]

    IS_COLAB = True
except ImportError:
    colab_files = None
    colab_eval_js = None
    IS_COLAB = False


def _unwrap_nested_project_dir(work_dir: Path) -> Path:
    """Handle both flat zip (src/ at root) and single-folder wrapped zip (Chip_detection/src/)."""
    if (work_dir / "src" / "inference" / "inspector.py").exists():
        return work_dir
    for sub in work_dir.iterdir():
        if sub.is_dir() and (sub / "src" / "inference" / "inspector.py").exists():
            return sub
    return work_dir


def setup_colab_workspace() -> tuple[Path, str]:
    """Set up workspace on Google Colab (with interactive upload prompt) or Local Python."""
    if IS_COLAB:
        content_root = Path(os.environ.get("COLAB_CONTENT_ROOT", "/content"))
        work_dir = content_root / "Chip_detection"

        existing_resolved = _unwrap_nested_project_dir(work_dir) if work_dir.exists() else work_dir
        if not (existing_resolved / "src" / "inference" / "inspector.py").exists():
            # 1. Kiểm tra xem người dùng đã kéo thả sẵn file .zip vào /content hoặc Google Drive chưa
            candidate_zips = [
                content_root / "colab_bundle.zip",
                content_root / "drive" / "MyDrive" / "colab_bundle.zip",
            ]
            candidate_zips.extend(sorted(content_root.glob("*.zip")))
            bundle_zip: Path | None = next((p for p in candidate_zips if p.is_file()), None)

            # 2. Nếu chưa có sẵn file .zip thì hiện nút Choose Files ngay trong Cell Output để tải lên
            if bundle_zip is None and colab_files is not None:
                display(
                    HTML(
                        '<div style="background:#eff6ff;border:2px solid #3b82f6;border-radius:8px;'
                        'padding:12px 16px;margin-bottom:8px;color:#1e3a8a;font-family:sans-serif;">'
                        "📂 <b>Bước 1:</b> Hãy bấm nút <b>Choose Files</b> bên dưới và chọn file "
                        "<code>colab_bundle.zip</code> từ máy tính của bạn (chờ thanh tiến trình đạt 100%)."
                        "</div>"
                    )
                )
                uploaded = colab_files.upload()
                if uploaded:
                    uploaded_name = next(iter(uploaded.keys()))
                    cand = Path(uploaded_name)
                    bundle_zip = cand if cand.is_file() else (content_root / cand.name)
                    if not bundle_zip.is_file():
                        bundle_zip = content_root / cand.name
                        bundle_zip.write_bytes(uploaded[uploaded_name])

            if bundle_zip is None or not bundle_zip.is_file():
                raise FileNotFoundError(
                    "Chưa nhận được file 'colab_bundle.zip'. "
                    "Hãy chạy lại ô này và bấm 'Choose Files' để chọn file 'colab_bundle.zip'!"
                )

            print(f"📦 Đang giải nén '{bundle_zip.name}' vào {work_dir}...")
            if work_dir.exists():
                shutil.rmtree(work_dir)
            work_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(bundle_zip, "r") as zf:
                zf.extractall(work_dir)

        final_root = _unwrap_nested_project_dir(work_dir)
        os.chdir(final_root)
        return final_root, "Google Colab"

    # Chạy ở máy Local (Jupyter / VS Code)
    cwd = Path.cwd().resolve()
    if (cwd / "src" / "inference" / "inspector.py").exists():
        return cwd, "Local Python"
    if cwd.name == "notebooks" and (cwd.parent / "src" / "inference" / "inspector.py").exists():
        os.chdir(cwd.parent)
        return cwd.parent, "Local Python"

    raise FileNotFoundError("Không tìm thấy thư mục 'src/'. Hãy đặt notebook trong dự án hoặc chạy trên Google Colab.")


PROJECT_ROOT, ENV_NAME = setup_colab_workspace()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.acquisition.camera_trigger import CameraTrigger, TriggerState
from src.acquisition.image_loader import load_image_from_bytes, load_image_from_path
from src.inference.inspector import InspectionModel, update_threshold_decision
from src.preprocessing.board_aligner import BoardAligner, preprocess_board_image
from src.schemas import InspectionResult, PreprocessResult
from src.training.evaluate import evaluate_dataset
from src.ui.app import (
    draw_board_bbox_overlay,
    draw_camera_roi_overlay,
    format_inspection_badge_html,
    format_trigger_status_text,
    get_dataset_counts,
    recalibrate_history_entries,
)

# Khởi tạo và nạp trọng số mô hình PatchCore + PinInspector
inspector = InspectionModel(
    weights_path=PROJECT_ROOT / "weights" / "patchcore_best.ckpt",
    metadata_path=PROJECT_ROOT / "weights" / "metadata.json",
    config_path=PROJECT_ROOT / "configs" / "patchcore_chip.yaml",
)
inspector.load()

camera_trigger = CameraTrigger.from_config(PROJECT_ROOT / "configs" / "patchcore_chip.yaml")
ds_counts = get_dataset_counts(PROJECT_ROOT / "data" / "chip_dataset")
runtime_device = f"CUDA GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU"
backend_type = getattr(inspector._feature_extractor, "backend_type", "unknown")
mem_shape = inspector._memory_bank.shape if inspector._memory_bank is not None else (0, 0)

display(
    HTML(
        f"""
        <div style="background:#f0fdf4;border:2px solid #16a34a;border-radius:8px;padding:14px 18px;font-family:sans-serif;color:#14532d;">
            <div style="font-size:1.15rem;font-weight:700;margin-bottom:6px;">
                ✅ Hệ thống Kiểm tra Bo mạch LM393 đã sẵn sàng trên {ENV_NAME}!
            </div>
            <div style="font-size:0.95rem;line-height:1.6;">
                • <b>Thư mục làm việc:</b> <code>{PROJECT_ROOT}</code><br>
                • <b>Thiết bị tính toán:</b> {runtime_device} | <b>Feature Backend:</b> <code>{backend_type}</code> (<code>{inspector.backbone_name}</code>)<br>
                • <b>PatchCore Memory Bank:</b> {mem_shape[0]} coreset patches × {mem_shape[1]} chiều | <b>Ngưỡng chuẩn (Threshold τ):</b> <code>{inspector.default_threshold:.4f}</code><br>
                • <b>Bộ dữ liệu mẫu đi kèm:</b> {ds_counts['normal_train']} Normal Train · {ds_counts['normal_val']} Normal Val · {ds_counts['abnormal_test']} Abnormal Test
            </div>
        </div>
        """
    )
)

## 2. Hàm Kiểm tra & Hiển thị Trực quan 4 Khung hình (`render_inspection_report`)

Mỗi lần kiểm tra một bo mạch, hệ thống hiển thị trực tiếp ngay trong ô kết quả:
1. **Bảng kết luận `TRUE — NORMAL` (Xanh) hoặc `FALSE — ANOMALY` (Đỏ)** kèm điểm bất thường `Anomaly Score`, ngưỡng `Threshold τ`, kết quả bề mặt mạch (`Board`) và kết quả 3 chân cắm (`3 Chân`).
2. **4 khung hình đối chiếu trực quan**:
   - **Khung 1**: Ảnh gốc đóng khung `Full ROI` (xanh lá) và `Thân PCB` (xanh dương).
   - **Khung 2**: Ảnh sau căn chỉnh góc xoay, xóa bóng đổ và cân bằng sáng CLAHE (`256×256`).
   - **Khung 3**: Bản đồ nhiệt `Anomaly Heatmap` từ PatchCore (vùng Đỏ/Vàng chỉ vị trí linh kiện bất thường).
   - **Khung 4**: Ảnh phóng đại vùng 3 chân cắm `G - V+ - S` (`256×256`) kèm bảng đo kích thước từng chân.

In [ ]:
# ==============================================================================
# 2. HÀM HIỂN THỊ TRỰC QUAN 4 KHUNG HÌNH & ĐỐI CHIẾU MẪU NORMAL VS ANOMALY
# ==============================================================================
def render_inspection_report(
    raw_rgb: np.ndarray,
    prep_result: PreprocessResult,
    insp_result: InspectionResult,
    title_prefix: str = "",
) -> None:
    """Render HTML status badge, 4-panel visual breakdown, and 3-pin measurement table."""
    if title_prefix:
        display(
            HTML(
                f'<div style="font-size:1.05rem;font-weight:700;margin:10px 0 6px 0;color:#1e293b;">'
                f"📌 {title_prefix}</div>"
            )
        )

    display(HTML(format_inspection_badge_html(insp_result)))

    warnings_list = list(getattr(insp_result, "warnings", None) or getattr(prep_result, "warnings", None) or [])
    for warn_text in warnings_list:
        display(
            HTML(
                f'<div style="background:#fef9c3;border-left:4px solid #eab308;padding:8px 12px;'
                f'margin-bottom:8px;color:#854d0e;font-size:0.9rem;">⚠️ <b>Cảnh báo tiền xử lý:</b> {warn_text}</div>'
            )
        )

    fig, axes = plt.subplots(1, 4, figsize=(18, 4.6), dpi=110)

    # Panel 1: Ảnh gốc + khung phát hiện bo mạch
    boxed_rgb = draw_board_bbox_overlay(
        raw_rgb,
        prep_result.bbox_box_points,
        pcb_bbox_points=getattr(prep_result, "pcb_bbox_points", None),
    )
    axes[0].imshow(boxed_rgb)
    axes[0].set_title("1. Ảnh gốc & Khung ROI\n(Xanh lá: Full ROI | Xanh dương: PCB)", fontsize=10, fontweight="bold")
    axes[0].axis("off")

    # Panel 2: Ảnh sau tiền xử lý 256x256
    if prep_result.normalized_rgb is not None:
        margin_pct = float(getattr(prep_result, "pin_end_margin", 0.0) or 0.0) * 100.0
        axes[1].imshow(prep_result.normalized_rgb)
        axes[1].set_title(
            f"2. Sau Tiền xử lý (256×256)\nXoay {prep_result.rotation_angle:.1f}° | CLAHE | Pin Margin {margin_pct:.1f}%",
            fontsize=10,
            fontweight="bold",
        )
    else:
        axes[1].text(0.5, 0.5, "Không tách được bo mạch", ha="center", va="center", fontsize=11)
        axes[1].set_title("2. Sau Tiền xử lý", fontsize=10, fontweight="bold")
    axes[1].axis("off")

    # Panel 3: Bản đồ nhiệt PatchCore
    if insp_result.heatmap_overlay_rgb is not None:
        axes[2].imshow(insp_result.heatmap_overlay_rgb)
        score_txt = f"{insp_result.anomaly_score:.4f}" if insp_result.anomaly_score is not None else "N/A"
        axes[2].set_title(
            f"3. Bản đồ nhiệt PatchCore\nScore: {score_txt} (Ngưỡng τ: {insp_result.threshold:.4f})",
            fontsize=10,
            fontweight="bold",
        )
    else:
        axes[2].text(0.5, 0.5, "Chưa có Heatmap", ha="center", va="center", fontsize=11)
        axes[2].set_title("3. Bản đồ nhiệt PatchCore", fontsize=10, fontweight="bold")
    axes[2].axis("off")

    # Panel 4: Kiểm tra chuyên biệt 3 chân cắm
    pin_res = getattr(insp_result, "pin_result", None)
    if pin_res is not None and pin_res.overlay_rgb is not None:
        axes[3].imshow(pin_res.overlay_rgb)
        pin_state_lbl = "ĐẠT (OK)" if pin_res.pins_ok else "LỖI (NG)"
        axes[3].set_title(
            f"4. Soi 3 Chân cắm ({pin_res.pin_count}/3 chân)\nKết luận chân: {pin_state_lbl}",
            fontsize=10,
            fontweight="bold",
        )
    else:
        axes[3].text(0.5, 0.5, "Không có dữ liệu 3 chân", ha="center", va="center", fontsize=11)
        axes[3].set_title("4. Soi 3 Chân cắm", fontsize=10, fontweight="bold")
    axes[3].axis("off")

    plt.tight_layout()
    plt.show()

    # Bảng chi tiết thông số đo 3 chân cắm
    if pin_res is not None and pin_res.per_pin:
        pin_df = pd.DataFrame(
            [
                {
                    "Chân cắm": p.get("pin_name", ""),
                    "Trạng thái": "✅ OK" if p.get("status") == "OK" else "❌ NG",
                    "Chiều dài (px)": p.get("length_px", 0.0),
                    "Độ rộng (px)": p.get("width_px", 0.0),
                    "Góc lệch (°)": p.get("angle_deg", 0.0),
                    "Khoảng cách (px)": p.get("spacing_px", "-"),
                    "Chi tiết đánh giá": p.get("reason_str", "OK"),
                }
                for p in pin_res.per_pin
            ]
        )
        display(pin_df)


# Chạy minh họa ngay 1 mẫu Chuẩn (Normal) và 1 mẫu Lỗi (Anomaly) để đối chiếu trực tiếp
for demo_title, rel_sample in [
    ("Mẫu Đối Chứng 1: Bo mạch Chuẩn (Normal — Đủ linh kiện & 3 chân thẳng)", "data/chip_dataset/normal/val/normal_val_001.jpg"),
    ("Mẫu Đối Chứng 2: Bo mạch Lỗi Bề Mặt (Anomaly — Che khuất biến trở 103)", "data/chip_dataset/abnormal/test/anomaly_test_005.jpg"),
]:
    sample_file = PROJECT_ROOT / rel_sample
    if sample_file.exists():
        demo_rgb = load_image_from_path(sample_file)
        demo_prep = preprocess_board_image(demo_rgb, aligner=inspector.aligner)
        demo_res = inspector.predict(demo_rgb, preprocess_result=demo_prep)
        render_inspection_report(demo_rgb, demo_prep, demo_res, title_prefix=f"{demo_title} [{rel_sample}]")

## 3. Kiểm định Tự động Toàn bộ Tập Validation & Test (`22 ảnh thực tế`)

Ô dưới đây chạy đánh giá hàng loạt trên `10` ảnh mạch chuẩn (`normal/val`) và `12` ảnh mạch lỗi thực tế (`abnormal/test`: thiếu SMD, che biến trở `103`, che IC `LM393`, che LED, cong/che chân cắm) để kiểm chứng các chỉ số `AUROC`, `F1-Score` và `Confusion Matrix`.

In [ ]:
# ==============================================================================
# 3. CHẠY KIỂM ĐỊNH ĐỘ CHÍNH XÁC TRÊN TOÀN BỘ TẬP VALIDATION & TEST
# ==============================================================================
eval_report = evaluate_dataset(
    config_path=PROJECT_ROOT / "configs" / "patchcore_chip.yaml",
    weights_path=PROJECT_ROOT / "weights" / "patchcore_best.ckpt",
    dataset_root=PROJECT_ROOT / "data" / "chip_dataset",
    metadata_path=PROJECT_ROOT / "weights" / "metadata.json",
    update_metadata=False,
)

cm = eval_report["confusion_matrix"]
display(
    HTML(
        f"""
        <div style="background:#eff6ff;border:2px solid #2563eb;border-radius:8px;padding:12px 16px;margin-bottom:10px;color:#1e3a8a;">
            <b>📊 Kết quả Kiểm định Tổng thể ({eval_report['num_normal_val']} Normal Val + {eval_report['num_abnormal_test']} Abnormal Test):</b><br>
            • <b>AUROC:</b> <code>{eval_report['auroc']:.4f}</code> &nbsp;|&nbsp;
            <b>F1-Score:</b> <code>{eval_report['f1_score']:.4f}</code> &nbsp;|&nbsp;
            <b>Accuracy:</b> <code>{eval_report['accuracy'] * 100:.1f}%</code> &nbsp;|&nbsp;
            <b>Threshold tối ưu (τ):</b> <code>{eval_report['optimal_threshold']:.4f}</code><br>
            • <b>Confusion Matrix:</b> True Negative (Normal đúng) = <code>{cm['tn']}</code> ·
            False Positive (Báo nhầm) = <code>{cm['fp']}</code> ·
            True Positive (Bắt đúng lỗi) = <code>{cm['tp']}</code> ·
            False Negative (Bỏ lọt lỗi) = <code>{cm['fn']}</code>
        </div>
        """
    )
)

detailed_rows: list[dict[str, object]] = []
for subset_label, folder in [
    ("Normal", PROJECT_ROOT / "data" / "chip_dataset" / "normal" / "val"),
    ("Anomaly", PROJECT_ROOT / "data" / "chip_dataset" / "abnormal" / "test"),
]:
    for img_p in sorted(folder.glob("*.jpg")):
        rgb_i = load_image_from_path(img_p)
        res_i = inspector.predict(rgb_i)
        pred_label = "Normal" if res_i.is_normal else "Anomaly"
        detailed_rows.append(
            {
                "Tên file": img_p.name,
                "Nhãn thực tế": subset_label,
                "Dự đoán": pred_label,
                "Khớp": "✅" if pred_label == subset_label else "❌",
                "Anomaly Score": res_i.anomaly_score,
                "Threshold τ": res_i.threshold,
                "Bề mặt PCB": "OK" if res_i.board_ok else "FAIL",
                "3 Chân cắm": "OK" if res_i.pins_ok else "FAIL",
                "Thời gian (ms)": res_i.processing_time_ms,
                "Thông báo chi tiết": res_i.message,
            }
        )

eval_df = pd.DataFrame(detailed_rows)
display(eval_df)

## 4. Bảng Điều Khiển Kiểm Tra Tương Tác (Phase 1 — Chọn mẫu / Tải ảnh mới lên / Tinh chỉnh nóng $\tau$)

- Chọn nhanh bất kỳ ảnh mẫu nào từ danh sách thả xuống **hoặc bấm nút `📤 Tải ảnh từ máy tính lên`**.
- Kéo thanh trượt **`Threshold τ`** để cập nhật tức thì kết quả `TRUE/FALSE` ($O(1)$) mà **không cần chạy lại mạng nơ-ron**.
- Điều chỉnh **`Độ nhạy chân`** hoặc **`CLAHE Clip`** để kiểm thử độ bền dưới các điều kiện ánh sáng khác nhau.

In [ ]:
# ==============================================================================
# 4. BẢNG ĐIỀU KHIỂN TƯƠNG TÁC PHASE 1 (CHỌN MẪU / COLAB UPLOAD / LIVE SLIDER)
# ==============================================================================
import ipywidgets as widgets


def build_sample_options() -> list[tuple[str, str]]:
    """Build labeled list of curated + all dataset sample paths."""
    curated = [
        ("🟢 [Normal] Mạch chuẩn thực tế (normal_val_001.jpg)", "data/chip_dataset/normal/val/normal_val_001.jpg"),
        ("🔴 [Anomaly] Che biến trở 103 (anomaly_test_005.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_005.jpg"),
        ("🔴 [Anomaly] Che IC LM393 (anomaly_test_007.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_007.jpg"),
        ("🔴 [Anomaly] Che đèn LED (anomaly_test_001.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_001.jpg"),
        ("🔴 [Anomaly] Cong chân cắm header (anomaly_test_011.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_011.jpg"),
        ("🔴 [Anomaly] Dán che vùng chân cắm (anomaly_test_003.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_003.jpg"),
        ("🟢 [Normal] Mẫu gốc chuẩn (image/sample2.jpg)", "image/sample2.jpg"),
        ("🔴 [Anomaly] Mẫu gốc thiếu linh kiện SMD (image/sample1.jpg)", "image/sample1.jpg"),
    ]
    seen: set[str] = set()
    options: list[tuple[str, str]] = []
    for lbl, rel in curated:
        if (PROJECT_ROOT / rel).exists():
            options.append((lbl, rel))
            seen.add(rel)

    for sub_tag, folder_rel in [
        ("🟢 [Normal Val]", "data/chip_dataset/normal/val"),
        ("🔴 [Abnormal Test]", "data/chip_dataset/abnormal/test"),
    ]:
        folder = PROJECT_ROOT / folder_rel
        if folder.exists():
            for p in sorted(folder.glob("*.jpg")):
                rel_str = p.relative_to(PROJECT_ROOT).as_posix()
                if rel_str not in seen:
                    options.append((f"{sub_tag} {p.name}", rel_str))
                    seen.add(rel_str)
    return options


sample_dropdown = widgets.Dropdown(
    options=build_sample_options(),
    description="Ảnh mẫu:",
    layout=widgets.Layout(width="62%"),
    style={"description_width": "90px"},
)

colab_upload_btn = widgets.Button(
    description="📤 Tải ảnh từ máy tính lên",
    button_style="info",
    layout=widgets.Layout(width="35%", height="34px"),
)

tau_slider = widgets.FloatSlider(
    value=round(float(inspector.default_threshold), 2),
    min=0.0,
    max=1.0,
    step=0.01,
    description="Threshold τ:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="48%"),
    style={"description_width": "95px"},
)

pin_sens_slider = widgets.FloatSlider(
    value=1.0,
    min=0.5,
    max=2.0,
    step=0.05,
    description="Độ nhạy chân:",
    continuous_update=False,
    readout_format=".2f",
    layout=widgets.Layout(width="48%"),
    style={"description_width": "95px"},
)

clahe_chk = widgets.Checkbox(
    value=True,
    description="Bật cân bằng sáng CLAHE & White Balance",
    indent=False,
    layout=widgets.Layout(width="48%"),
)

clahe_clip_slider = widgets.FloatSlider(
    value=1.5,
    min=0.5,
    max=4.0,
    step=0.1,
    description="CLAHE Clip:",
    continuous_update=False,
    readout_format=".1f",
    layout=widgets.Layout(width="48%"),
    style={"description_width": "95px"},
)

run_btn = widgets.Button(
    description="🔍 Phân tích lại ảnh hiện tại",
    button_style="primary",
    layout=widgets.Layout(width="260px", height="36px"),
)

upload_prompt_out = widgets.Output()
phase1_out = widgets.Output()
_phase1_state: dict[str, object] = {
    "mode": "sample",  # "sample" or "uploaded"
    "uploaded_name": "",
    "uploaded_rgb": None,
}


def _execute_phase1(recompute_inference: bool = True) -> None:
    with phase1_out:
        clear_output(wait=True)

        if _phase1_state.get("mode") == "uploaded" and _phase1_state.get("uploaded_rgb") is not None:
            raw_rgb = _phase1_state["uploaded_rgb"]
            assert isinstance(raw_rgb, np.ndarray)
            fname = str(_phase1_state.get("uploaded_name", "uploaded.jpg"))
            source_label = f"Ảnh tải lên từ máy: {fname}"
            source_key = f"upload:{fname}:{raw_rgb.shape}"
        else:
            rel_path = str(sample_dropdown.value)
            raw_rgb = load_image_from_path(PROJECT_ROOT / rel_path)
            source_label = f"Ảnh mẫu: {rel_path}"
            source_key = rel_path

        sig = f"{source_key}|{clahe_chk.value}|{clahe_clip_slider.value:.2f}|{pin_sens_slider.value:.2f}"
        if recompute_inference or _phase1_state.get("sig") != sig:
            if isinstance(inspector.aligner, BoardAligner):
                inspector.aligner.apply_clahe = bool(clahe_chk.value)
                inspector.aligner.apply_clahe_flag = bool(clahe_chk.value)
                inspector.aligner.clahe_clip_limit = float(clahe_clip_slider.value)

            prep_res = preprocess_board_image(raw_rgb, aligner=inspector.aligner)
            insp_res = inspector.predict(
                raw_rgb,
                threshold=float(tau_slider.value),
                preprocess_result=prep_res,
                pin_sensitivity=float(pin_sens_slider.value),
            )
            _phase1_state["sig"] = sig
            _phase1_state["raw_rgb"] = raw_rgb
            _phase1_state["prep_res"] = prep_res
            _phase1_state["insp_res"] = insp_res
            _phase1_state["source_label"] = source_label
        else:
            cached_res = _phase1_state["insp_res"]
            assert isinstance(cached_res, InspectionResult)
            _phase1_state["insp_res"] = update_threshold_decision(cached_res, float(tau_slider.value))

        render_inspection_report(
            _phase1_state["raw_rgb"],  # type: ignore[arg-type]
            _phase1_state["prep_res"],  # type: ignore[arg-type]
            _phase1_state["insp_res"],  # type: ignore[arg-type]
            title_prefix=str(_phase1_state["source_label"]),
        )


def _on_sample_dropdown_change(_change: object) -> None:
    _phase1_state["mode"] = "sample"
    _execute_phase1(recompute_inference=True)


def _on_colab_upload_click(_btn: object) -> None:
    with upload_prompt_out:
        clear_output(wait=True)
        if colab_files is None:
            print("Chức năng hộp thoại Colab Upload chỉ hoạt động trên Google Colab. Hãy dùng Cell 6 khi chạy Local.")
            return
        print("👉 Chọn file ảnh bo mạch (.jpg, .png) từ máy tính của bạn:")
        uploaded_dict = colab_files.upload()
        clear_output(wait=True)
    if uploaded_dict:
        fname, img_bytes = next(iter(uploaded_dict.items()))
        _phase1_state["mode"] = "uploaded"
        _phase1_state["uploaded_name"] = str(fname)
        _phase1_state["uploaded_rgb"] = load_image_from_bytes(img_bytes)
        _execute_phase1(recompute_inference=True)


sample_dropdown.observe(_on_sample_dropdown_change, names="value")
colab_upload_btn.on_click(_on_colab_upload_click)
tau_slider.observe(lambda _c: _execute_phase1(recompute_inference=False), names="value")
pin_sens_slider.observe(lambda _c: _execute_phase1(recompute_inference=True), names="value")
clahe_chk.observe(lambda _c: _execute_phase1(recompute_inference=True), names="value")
clahe_clip_slider.observe(lambda _c: _execute_phase1(recompute_inference=True), names="value")
run_btn.on_click(lambda _b: _execute_phase1(recompute_inference=True))

ui_box = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0 0 6px 0;'>🎛️ Bảng Điều Khiển Kiểm Tra Bo Mạch (Phase 1)</h4>"),
        widgets.HBox([sample_dropdown, colab_upload_btn]),
        upload_prompt_out,
        widgets.HBox([tau_slider, pin_sens_slider]),
        widgets.HBox([clahe_chk, clahe_clip_slider]),
        run_btn,
        phase1_out,
    ],
    layout=widgets.Layout(border="1px solid #cbd5e1", padding="14px", border_radius="8px"),
)

display(ui_box)
_execute_phase1(recompute_inference=True)

## 5. Kiểm tra qua Camera & Mô phỏng Auto-Trigger (Phase 2 — Webcam Trình duyệt + `CameraTrigger`)

Ở Phase 2, bạn có thể kiểm thử cả **2 chế độ**:
1. **Mô phỏng Máy trạng thái 4 bước (`CameraTrigger`)**: Kiểm chứng chu trình `IDLE_EMPTY` → `STABILIZING (10 frames đứng yên)` → `TRIGGER_CAPTURE` → `COOLDOWN_WAIT_EXIT` (chống chụp lặp cho tới khi nhấc mạch ra đủ `5 frames`).
2. **Chụp trực tiếp từ Webcam thật của máy tính (`🎥 Mở Webcam Máy Tính & Chụp`)**: Trình duyệt Chrome/Edge sẽ bật camera gắn trên máy bạn ngay trong ô Colab để chụp mạch thực tế và chấm điểm ngay lập tức.

In [ ]:
# ==============================================================================
# 5. PHASE 2: MÔ PHỎNG AUTO-TRIGGER & CHỤP WEBCAM THẬT TRÊN TRÌNH DUYỆT COLAB
# ==============================================================================
def capture_webcam_frame_on_colab(quality: float = 0.92) -> np.ndarray | None:
    """Open browser webcam inside Google Colab output cell and return captured RGB frame."""
    if colab_eval_js is None:
        print("Webcam trình duyệt chỉ hỗ trợ khi chạy trên Google Colab.")
        return None

    js_code = f"""
    async function captureColabWebcam() {{
      const container = document.createElement('div');
      container.style.cssText = 'margin:8px 0;padding:10px;border:2px solid #2563eb;border-radius:8px;background:#f8fafc;display:inline-block;';
      const video = document.createElement('video');
      video.style.cssText = 'display:block;max-width:520px;border-radius:6px;';
      const btnBar = document.createElement('div');
      btnBar.style.cssText = 'margin-top:8px;display:flex;gap:8px;';

      const capBtn = document.createElement('button');
      capBtn.textContent = '📸 Chụp & Kiểm tra mạch này';
      capBtn.style.cssText = 'background:#16a34a;color:white;border:none;padding:8px 16px;border-radius:6px;font-weight:bold;cursor:pointer;';

      const cancelBtn = document.createElement('button');
      cancelBtn.textContent = '✖ Hủy';
      cancelBtn.style.cssText = 'background:#64748b;color:white;border:none;padding:8px 14px;border-radius:6px;cursor:pointer;';

      btnBar.appendChild(capBtn);
      btnBar.appendChild(cancelBtn);
      container.appendChild(video);
      container.appendChild(btnBar);
      document.body.appendChild(container);

      const stream = await navigator.mediaDevices.getUserMedia({{video: {{facingMode: 'environment'}}}});
      video.srcObject = stream;
      await video.play();

      const action = await new Promise((resolve) => {{
        capBtn.onclick = () => resolve('capture');
        cancelBtn.onclick = () => resolve('cancel');
      }});

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);

      stream.getVideoTracks()[0].stop();
      container.remove();
      if (action !== 'capture') return '';
      return canvas.toDataURL('image/jpeg', {quality});
    }}
    captureColabWebcam();
    """
    data_url = colab_eval_js(js_code)
    if not data_url or "," not in str(data_url):
        return None
    raw_bytes = base64.b64decode(str(data_url).split(",", 1)[1])
    return load_image_from_bytes(raw_bytes)


sim_board_dropdown = widgets.Dropdown(
    options=[
        ("🟢 Đặt mạch Normal vào ROI (normal_val_001.jpg)", "data/chip_dataset/normal/val/normal_val_001.jpg"),
        ("🔴 Đặt mạch Lỗi Che biến trở 103 vào ROI (anomaly_test_005.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_005.jpg"),
        ("🔴 Đặt mạch Lỗi Che IC LM393 vào ROI (anomaly_test_007.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_007.jpg"),
        ("🔴 Đặt mạch Lỗi Cong chân cắm vào ROI (anomaly_test_011.jpg)", "data/chip_dataset/abnormal/test/anomaly_test_011.jpg"),
    ],
    description="Kịch bản:",
    layout=widgets.Layout(width="70%"),
    style={"description_width": "85px"},
)

btn_auto_trigger = widgets.Button(
    description="▶️ Mô phỏng Auto-Trigger (10 frames)",
    button_style="success",
    layout=widgets.Layout(width="265px"),
)
btn_webcam_live = widgets.Button(
    description="🎥 Mở Webcam Máy Tính & Chụp",
    button_style="info",
    layout=widgets.Layout(width="250px"),
)
btn_manual_cap = widgets.Button(
    description="📸 Chụp Thủ Công (Manual)",
    button_style="warning",
    layout=widgets.Layout(width="210px"),
)
btn_remove_board = widgets.Button(
    description="👋 Nhấc mạch (Reset Cooldown)",
    button_style="",
    layout=widgets.Layout(width="225px"),
)

phase2_out = widgets.Output()
phase2_session_history: list[dict[str, object]] = []


def _run_phase2_action(action: str) -> None:
    with phase2_out:
        clear_output(wait=True)
        rel_p = str(sim_board_dropdown.value)
        board_frame = load_image_from_path(PROJECT_ROOT / rel_p)
        empty_frame = np.full_like(board_frame, 242)

        logs: list[str] = []
        captured_frame: np.ndarray | None = None
        last_info: dict[str, object] = {"state": camera_trigger.state.value}
        display_frame = board_frame
        source_name = Path(rel_p).name
        mode_tag = "Auto-Trigger"

        if action == "webcam":
            cam_rgb = capture_webcam_frame_on_colab()
            if cam_rgb is None:
                print("Đã hủy hoặc không lấy được khung hình từ Webcam.")
                return
            captured_frame = camera_trigger.force_manual_trigger(cam_rgb)
            display_frame = cam_rgb
            last_info = {"state": camera_trigger.state.value}
            source_name = "Webcam_Capture"
            mode_tag = "Colab Webcam"
            logs.append("🎥 Đã chụp khung hình trực tiếp từ Webcam trình duyệt trên Google Colab!")

        elif action == "auto":
            if camera_trigger.state == TriggerState.COOLDOWN_WAIT_EXIT:
                logs.append("• Đang ở COOLDOWN_WAIT_EXIT -> Mô phỏng nhấc mạch cũ ra (5 frames trống)...")
                for _ in range(camera_trigger.cooldown_empty_frames):
                    camera_trigger.process_frame(empty_frame)
                logs.append(f"  -> Đã trở về {camera_trigger.state.value}")

            rng = np.random.default_rng(7)
            for step_idx in range(1, camera_trigger.stable_frames_required + 3):
                if step_idx <= 2:
                    shift_x = int(step_idx * 14)
                    mat = np.float32([[1, 0, shift_x], [0, 1, 0]])
                    frame_step = cv2.warpAffine(
                        board_frame,
                        mat,
                        (board_frame.shape[1], board_frame.shape[0]),
                        borderValue=(240, 240, 240),
                    )
                else:
                    jitter = rng.integers(-1, 2, size=board_frame.shape, dtype=np.int16)
                    frame_step = np.clip(board_frame.astype(np.int16) + jitter, 0, 255).astype(np.uint8)

                state, should_trig, cap_f, last_info = camera_trigger.process_frame(frame_step)
                logs.append(
                    f"Frame {step_idx:02d}: state={state.value} | "
                    f"stable={last_info['stable_counter']}/{last_info['required_stable']} | "
                    f"movement={last_info['movement']} px"
                )
                display_frame = frame_step
                if should_trig and cap_f is not None:
                    captured_frame = cap_f
                    logs.append("🎯 ĐÃ KÍCH HOẠT AUTO-TRIGGER! Chọn frame nét nhất để đưa vào PatchCore & PinInspector.")
                    camera_trigger.process_frame(frame_step)
                    break

        elif action == "manual":
            captured_frame = camera_trigger.force_manual_trigger(board_frame)
            last_info = {"state": camera_trigger.state.value}
            display_frame = board_frame
            mode_tag = "Manual"
            logs.append(f"📸 Đã bấm Manual Capture -> Chuyển sang {camera_trigger.state.value}.")

        elif action == "remove":
            for idx in range(1, camera_trigger.cooldown_empty_frames + 1):
                state, _, _, last_info = camera_trigger.process_frame(empty_frame)
                logs.append(f"Empty Frame {idx:02d}: state={state.value} | empty_counter={last_info['empty_counter']}")
            display_frame = empty_frame

        roi_vis = draw_camera_roi_overlay(display_frame, camera_trigger, last_info)
        fig_cam, ax_cam = plt.subplots(1, 1, figsize=(6.5, 4.5), dpi=105)
        ax_cam.imshow(roi_vis)
        ax_cam.set_title(
            f"Khung ngắm Camera ROI — Trạng thái: {format_trigger_status_text(camera_trigger.state, last_info)}",
            fontsize=10,
            fontweight="bold",
        )
        ax_cam.axis("off")
        plt.tight_layout()
        plt.show()

        print("\n".join(logs))

        if captured_frame is not None:
            prep_c = preprocess_board_image(captured_frame, aligner=inspector.aligner)
            insp_c = inspector.predict(
                captured_frame,
                threshold=float(tau_slider.value),
                preprocess_result=prep_c,
                pin_sensitivity=float(pin_sens_slider.value),
            )
            if insp_c.status == "OK" and insp_c.anomaly_score is not None:
                phase2_session_history.insert(
                    0,
                    {
                        "timestamp": datetime.now().strftime("%H:%M:%S"),
                        "source": source_name,
                        "mode": mode_tag,
                        "anomaly_score": insp_c.anomaly_score,
                        "threshold": insp_c.threshold,
                        "boolean_result": bool(insp_c.is_normal),
                        "result_label": "TRUE (Normal)" if insp_c.is_normal else "FALSE (Anomaly)",
                        "processing_time_ms": insp_c.processing_time_ms,
                    },
                )
            render_inspection_report(
                captured_frame,
                prep_c,
                insp_c,
                title_prefix=f"Kết quả Kiểm tra từ Camera ({mode_tag})",
            )

        if phase2_session_history:
            updated_hist = recalibrate_history_entries(phase2_session_history, float(tau_slider.value))
            display(HTML("<b>🗂️ Lịch sử Kiểm tra trong Phiên (Session History):</b>"))
            display(pd.DataFrame(updated_hist))


btn_auto_trigger.on_click(lambda _b: _run_phase2_action("auto"))
btn_webcam_live.on_click(lambda _b: _run_phase2_action("webcam"))
btn_manual_cap.on_click(lambda _b: _run_phase2_action("manual"))
btn_remove_board.on_click(lambda _b: _run_phase2_action("remove"))

phase2_box = widgets.VBox(
    [
        widgets.HTML("<h4 style='margin:0 0 6px 0;'>📷 Kiểm tra Camera & Máy trạng thái Auto-Trigger (Phase 2)</h4>"),
        sim_board_dropdown,
        widgets.HBox([btn_auto_trigger, btn_webcam_live, btn_manual_cap, btn_remove_board]),
        phase2_out,
    ],
    layout=widgets.Layout(border="1px solid #cbd5e1", padding="14px", border_radius="8px"),
)

display(phase2_box)
_run_phase2_action("auto")

## 6. Tải lên & Kiểm tra Hàng loạt Nhiều Ảnh Mới từ Máy Tính (Tùy chọn)

Khi team Kỹ thuật Điện chụp nhiều ảnh bo mạch mới bên ngoài và muốn tải lên kiểm tra cùng lúc **nhiều file ảnh**, chỉ cần bấm chạy riêng ô dưới đây:

In [ ]:
# ==============================================================================
# 6. TẢI LÊN 1 HOẶC NHIỀU ẢNH TỪ MÁY TÍNH ĐỂ KIỂM TRA HÀNG LOẠT (TÙY CHỌN)
# ==============================================================================
# Mặc định kiểm tra thử mẫu lỗi cong chân (anomaly_test_011.jpg).
# Đổi UPLOAD_NEW_IMAGES_NOW = True rồi chạy lại ô này nếu muốn mở hộp thoại chọn nhiều ảnh từ máy tính!
UPLOAD_NEW_IMAGES_NOW = False

if UPLOAD_NEW_IMAGES_NOW and colab_files is not None:
    print("👉 Hãy chọn 1 hoặc nhiều ảnh bo mạch (.jpg, .png) từ máy tính của bạn:")
    batch_uploaded = colab_files.upload()
    for fname, raw_bytes in batch_uploaded.items():
        u_rgb = load_image_from_bytes(raw_bytes)
        u_prep = preprocess_board_image(u_rgb, aligner=inspector.aligner)
        u_res = inspector.predict(
            u_rgb,
            threshold=float(tau_slider.value),
            preprocess_result=u_prep,
            pin_sensitivity=float(pin_sens_slider.value),
        )
        render_inspection_report(u_rgb, u_prep, u_res, title_prefix=f"Ảnh tải lên: {fname}")
else:
    demo_pin_defect = PROJECT_ROOT / "data" / "chip_dataset" / "abnormal" / "test" / "anomaly_test_011.jpg"
    if demo_pin_defect.exists():
        p_rgb = load_image_from_path(demo_pin_defect)
        p_prep = preprocess_board_image(p_rgb, aligner=inspector.aligner)
        p_res = inspector.predict(p_rgb, preprocess_result=p_prep)
        render_inspection_report(
            p_rgb,
            p_prep,
            p_res,
            title_prefix="Mẫu Đối Chứng 3: Bo mạch Lỗi Cong Chân Cắm [anomaly_test_011.jpg]",
        )